# Notebook 12 — Capstone: Voice-Driven Visual RAG Agent

Everything we've built, fused into one agent.

```
user voice question (any language, possibly code-switched)
        ↓        nb06/07 — Whisper, code-switching aware
    transcript
        ↓        nb03    — ColPali MaxSim over PDF page index
  top-K page images
        ↓        nb04    — strict schema + visual_evidence
  grounded answer
        ↓        — confidence gate
 safe-to-speak answer
        ↓        nb08    — TTS
    voice reply
```

Per-stage timing + token logging the whole way through, mockable agent layer for tests, refusal path on low-confidence answers. **This is what "production multimodal AI" looks like as a single artifact** — small enough to read in one sitting, every defensive design choice traceable to the section that motivates it.

## Prerequisites

- Drop a chart-heavy PDF at `../data/sample_pdfs/` (any of the ones you used in nb03/nb05).
- ColPali on MPS or CUDA (~6 GB).
- `OPENAI_API_KEY` in `.env`.

## 1. Load the heavy models once

In [ ]:
import asyncio, base64, io, json, logging, time
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

import numpy as np
import torch
from PIL import Image
from pdf2image import convert_from_path
from pydantic import BaseModel, Field, ConfigDict
from transformers import ColPaliForRetrieval, ColPaliProcessor
from faster_whisper import WhisperModel
from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
log = logging.getLogger("capstone")

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
log.info("device: %s", DEVICE)

openai_client = AsyncOpenAI()

log.info("loading Whisper...")
asr = WhisperModel("large-v3", device="cpu", compute_type="int8")

log.info("loading ColPali...")
colpali = ColPaliForRetrieval.from_pretrained(
    "vidore/colpali-v1.3-hf", torch_dtype=torch.bfloat16, device_map=DEVICE,
).eval()
colpali_proc = ColPaliProcessor.from_pretrained("vidore/colpali-v1.3-hf")
log.info("models ready")

## 2. Build the page index for the PDF

In [ ]:
PDF_DIR = Path("../data/sample_pdfs")
pdfs = sorted(PDF_DIR.glob("*.pdf"))
assert pdfs, f"drop a PDF into {PDF_DIR.resolve()}"
PDF = pdfs[0]
log.info("using %s", PDF.name)

MAX_PAGES = 30   # cap to keep the demo interactive
pages = convert_from_path(str(PDF), dpi=150, last_page=MAX_PAGES)
log.info("rendered %d pages", len(pages))

@torch.no_grad()
def embed_pages(pil_pages, batch=4):
    out = []
    for i in range(0, len(pil_pages), batch):
        b = pil_pages[i : i + batch]
        inputs = colpali_proc(images=b, return_tensors="pt").to(DEVICE)
        for emb in colpali(**inputs).embeddings:
            out.append(emb.cpu())
    return out

@torch.no_grad()
def embed_query(query: str):
    inputs = colpali_proc(text=[query], return_tensors="pt").to(DEVICE)
    return colpali(**inputs).embeddings[0].cpu()

def maxsim(q, p):
    return float((q.float() @ p.float().T).max(dim=1).values.sum())

page_embeddings = embed_pages(pages)
log.info("index ready: %d pages × ~%d patches each",
        len(page_embeddings), page_embeddings[0].shape[0])

## 3. The agent

One class. One method (`ask`). Every stage timed. Strict schema. Confidence gate. Refusal path. Token logging for FinOps.

Read it top to bottom — every defensive design choice maps to a section we covered.

In [ ]:
class GroundedAnswer(BaseModel):
    """From nb04 — the only difference is the page_number citation."""
    model_config = ConfigDict(extra="ignore")

    answer: Optional[str] = Field(None, description="Direct answer or null if not visible.")
    visual_evidence: str = Field(..., description="Verbatim quote / specific element observed.")
    page_number: Optional[int] = Field(None, description="Which provided page contained the answer.")
    confidence: float = Field(..., ge=0.0, le=1.0)


ANSWER_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "grounded_answer",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "required": ["answer", "visual_evidence", "page_number", "confidence"],
            "properties": {
                "answer":          {"type": ["string", "null"]},
                "visual_evidence": {"type": "string"},
                "page_number":     {"type": ["integer", "null"]},
                "confidence":      {"type": "number", "minimum": 0.0, "maximum": 1.0},
            },
        },
    },
}

GROUNDING = (
    "Answer the user's question using ONLY the provided PDF pages. "
    "Cite the page number you used. In `visual_evidence`, quote the exact "
    "text or describe the exact visual element you used (a specific chart "
    "bar, table cell, etc.) — generic descriptions are not acceptable. "
    "If the answer is not visible, set `answer` and `page_number` to null "
    "and `confidence` below 0.3."
)

GLOSSARY_FOR_CODESWITCH = (
    "PPT, ROI, KPI, OKR, EBITDA, Q1, Q2, Q3, Q4, revenue, margin, growth."
)


@dataclass
class StageTimings:
    stt_ms: float = 0.0
    retrieval_ms: float = 0.0
    llm_ms: float = 0.0
    tts_ms: float = 0.0
    total_ms: float = 0.0


@dataclass
class AgentResult:
    status: str                  # "ok" | "refused" | "no_match"
    transcript: str = ""
    answer: Optional[str] = None
    evidence: str = ""
    page: Optional[int] = None
    confidence: float = 0.0
    out_audio: Optional[Path] = None
    timings: StageTimings = field(default_factory=StageTimings)
    cost_usd: float = 0.0


class CapstoneAgent:
    CONFIDENCE_THRESHOLD = 0.4
    PRICING = {"in": 0.15 / 1e6, "out": 0.60 / 1e6}   # gpt-4o-mini

    def __init__(self, pages, page_embeddings, out_dir: Path):
        self.pages = pages
        self.page_embeddings = page_embeddings
        self.out_dir = out_dir
        self.out_dir.mkdir(parents=True, exist_ok=True)

    def _stt(self, audio_path: Path) -> tuple[str, str]:
        """Returns (transcript, language). Uses VAD chunking + glossary biasing
        to handle code-switched speech (nb07 strategy mix)."""
        segments_gen, info = asr.transcribe(
            str(audio_path),
            vad_filter=True,
            initial_prompt=GLOSSARY_FOR_CODESWITCH,
            condition_on_previous_text=False,
        )
        text = " ".join(s.text.strip() for s in segments_gen)
        return text, info.language

    def _retrieve(self, query: str, k: int = 3) -> list[int]:
        q = embed_query(query)
        scored = sorted(
            ((maxsim(q, pe), idx) for idx, pe in enumerate(self.page_embeddings)),
            reverse=True,
        )
        return [idx for _, idx in scored[:k]]

    def _page_to_b64(self, page_idx: int) -> str:
        buf = io.BytesIO(); self.pages[page_idx].save(buf, format="JPEG", quality=85)
        return base64.b64encode(buf.getvalue()).decode()

    async def _llm(self, query: str, page_indices: list[int]) -> tuple[GroundedAnswer, dict]:
        image_parts = []
        for idx in page_indices:
            image_parts.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{self._page_to_b64(idx)}",
                    "detail": "high",
                },
            })
        user_text = (
            f"Pages provided in order: {page_indices}\n\nQuestion: {query}"
        )
        r = await openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": GROUNDING},
                {"role": "user", "content": [*image_parts, {"type": "text", "text": user_text}]},
            ],
            response_format=ANSWER_SCHEMA,
        )
        parsed = GroundedAnswer.model_validate_json(r.choices[0].message.content)
        u = r.usage
        cost = u.prompt_tokens * self.PRICING["in"] + u.completion_tokens * self.PRICING["out"]
        return parsed, {"in": u.prompt_tokens, "out": u.completion_tokens, "cost": cost}

    async def _tts(self, text: str, out_path: Path):
        audio = await openai_client.audio.speech.create(model="tts-1", voice="nova", input=text)
        audio.stream_to_file(str(out_path))

    async def ask(self, audio_path: Path, name: str = "reply") -> AgentResult:
        timings = StageTimings()
        t_start = time.time()

        # STT
        t0 = time.time()
        transcript, lang = self._stt(audio_path)
        timings.stt_ms = (time.time() - t0) * 1000
        log.info("STT (%dms, lang=%s): %s", timings.stt_ms, lang, transcript[:80])

        if not transcript.strip():
            timings.total_ms = (time.time() - t_start) * 1000
            return AgentResult(status="no_match", transcript="", timings=timings)

        # Retrieval
        t0 = time.time()
        top_pages = self._retrieve(transcript, k=3)
        timings.retrieval_ms = (time.time() - t0) * 1000
        log.info("retrieval (%dms): pages %s", timings.retrieval_ms, top_pages)

        # LLM
        t0 = time.time()
        grounded, usage = await self._llm(transcript, top_pages)
        timings.llm_ms = (time.time() - t0) * 1000
        log.info("LLM  (%dms): conf=%.2f  page=%s  cost=$%.5f",
                 timings.llm_ms, grounded.confidence, grounded.page_number, usage["cost"])

        # Confidence gate (nb04)
        if grounded.confidence < self.CONFIDENCE_THRESHOLD or grounded.answer is None:
            timings.total_ms = (time.time() - t_start) * 1000
            log.info("refused: confidence below threshold")
            return AgentResult(
                status="refused",
                transcript=transcript,
                evidence=grounded.visual_evidence,
                confidence=grounded.confidence,
                timings=timings,
                cost_usd=usage["cost"],
            )

        # TTS
        t0 = time.time()
        out_audio = self.out_dir / f"{name}.mp3"
        await self._tts(grounded.answer, out_audio)
        timings.tts_ms = (time.time() - t0) * 1000

        timings.total_ms = (time.time() - t_start) * 1000
        return AgentResult(
            status="ok",
            transcript=transcript,
            answer=grounded.answer,
            evidence=grounded.visual_evidence,
            page=grounded.page_number,
            confidence=grounded.confidence,
            out_audio=out_audio,
            timings=timings,
            cost_usd=usage["cost"],
        )


agent = CapstoneAgent(
    pages=pages,
    page_embeddings=page_embeddings,
    out_dir=Path("../data/audio_samples/capstone"),
)
log.info("agent ready")

## 4. Generate test voice queries

Adapt the text below to questions actually answerable from your PDF — you'll get more useful results. The third query is deliberately code-switched (the realistic case).

In [ ]:
INPUT_DIR = Path("../data/audio_samples/capstone_input")
INPUT_DIR.mkdir(parents=True, exist_ok=True)

QUESTIONS = [
    ("q1_en",    "What is the title of this document?"),
    ("q2_zh",    "这份文件的主要主题是什么？"),
    ("q3_mixed", "PPT 第三页的 ROI 是多少？"),   # tweak to suit your PDF
    ("q4_bait",  "What is the CFO's signature on page 1?"),  # likely refused
]

from openai import OpenAI
sync_openai = OpenAI()
for name, text in QUESTIONS:
    p = INPUT_DIR / f"{name}.mp3"
    if not p.exists():
        sync_openai.audio.speech.create(model="tts-1", voice="nova", input=text).stream_to_file(str(p))
print("voice queries ready")

## 5. Run the agent end-to-end

In [ ]:
from IPython.display import Audio, display
from dataclasses import asdict

results = {}
for name, _ in QUESTIONS:
    in_path = INPUT_DIR / f"{name}.mp3"
    print(f"\n--- {name} ---")
    print("  user:")
    display(Audio(str(in_path)))
    res = await agent.ask(in_path, name=f"reply_{name}")
    results[name] = res
    print(f"  status:     {res.status}")
    print(f"  transcript: {res.transcript}")
    if res.status == "ok":
        print(f"  answer:     {res.answer}")
        print(f"  page:       {res.page}   confidence={res.confidence:.2f}")
        print(f"  evidence:   {res.evidence}")
        print("  agent reply:")
        display(Audio(str(res.out_audio)))
    else:
        print(f"  refused — {res.evidence}  (conf={res.confidence:.2f})")
    t = res.timings
    print(f"  timings: STT {t.stt_ms:.0f}ms  retr {t.retrieval_ms:.0f}ms  LLM {t.llm_ms:.0f}ms  TTS {t.tts_ms:.0f}ms  =  {t.total_ms:.0f}ms")
    print(f"  cost: ${res.cost_usd:.5f}")

## 6. The bait question — refusal in action

Look at the `q4_bait` row above. The PDF doesn't have a CFO signature on page 1 (probably) — the language prior says "corporate document → likely has a signature". The model could fabricate. The strict schema + visual_evidence + confidence gate combine to **refuse instead**.

That's the production-safety story in one row.

## 7. Aggregate timings + cost

In [ ]:
import matplotlib.pyplot as plt

names = list(results)
stt = [results[n].timings.stt_ms       for n in names]
ret = [results[n].timings.retrieval_ms for n in names]
llm = [results[n].timings.llm_ms       for n in names]
tts = [results[n].timings.tts_ms       for n in names]

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(names))
ax.bar(x, stt,                                              label="STT")
ax.bar(x, ret, bottom=stt,                                  label="Retrieval")
ax.bar(x, llm, bottom=np.array(stt) + np.array(ret),        label="LLM")
ax.bar(x, tts, bottom=np.array(stt) + np.array(ret) + np.array(llm), label="TTS")
ax.set_xticks(x); ax.set_xticklabels(names, rotation=20)
ax.set_ylabel("ms")
ax.set_title("Capstone agent — per-stage latency")
ax.legend()
plt.tight_layout(); plt.show()

total_cost = sum(r.cost_usd for r in results.values())
print(f"\nTotal cost across {len(results)} turns: ${total_cost:.5f}")
print("(scales linearly — for a 10K-turn day at this profile, that's ~${0:.2f})".format(total_cost * 10000 / len(results)))

## 8. Production checklist

Everything in this capstone matches a section from the original S3 deck.

**Visual:**
- ☑ ColPali retrieval over rendered PDF pages (§4.3)
- ☑ Strict-schema vision parsing with `visual_evidence` field (§7.3 / nb04)
- ☑ Confidence gate refusing low-confidence outputs (nb04)
- ☑ `detail=high` only because the queries can hit text inside charts (§2.1)

**Audio:**
- ☑ faster-whisper large-v3 with int8 quantization (§6.3)
- ☑ VAD chunking + glossary biasing for code-switched speech (nb07 strategy)
- ☑ Streaming-friendly architecture (each stage is async + measurable)

**Observability:**
- ☑ Per-stage timing (§10.4)
- ☑ Per-call token + cost logging (§10.2)
- ☑ Refusal events logged separately from successful answers

**What we didn't include (and where it'd go):**
- PII redaction on page images before sending — slot before `_page_to_b64`.
- Multimodal safety filters (Llama Guard 4 / ShieldGemma) on input + output — wrap `_llm` (§7.3 mitigation #5).
- Hallucination tracking — sample 1% of `ok` results for human review, store `answer + evidence + page` for offline scoring.
- Persistent index in Milvus 2.6+ (Array of Structs) instead of in-memory — see nb03 production notes.

## What we built (résumé bullet form)

*"Built a voice-driven multimodal RAG agent (Whisper + ColPali + GPT-4o + TTS) over chart-heavy PDFs, with multilingual code-switching support, strict-schema grounded generation and confidence gating to suppress fabrication, and per-stage observability. Demonstrated [N]× lower bad-data ingress vs vanilla VLM call in offline eval."*

Replace [N] when you build the eval set. The next step in this lab is exactly that — the eval suite.

## End